# wave 1 — prompt workshop (metaphor + parallel frames)

Steering-vector data for the extension in `extension/detailed plan v1.md`. Four experiments,
**94 concepts**, all fixed in `wave1_families.py` (items and correspondences are frozen there;
only *wording* — locks, avoid clauses, pool text — is workshopped here).

| dataset | $A$ (source) | $B$ (target) | shape | vectors |
|---|---|---|---|---|
| `size_imp_v1` (S3) | physical size ladder | importance ladder | ordered, rank-matched 9↔9 | 18 |
| `war_arg_v1` (S7) | literal war | argument / debate | categorical, 8 core + 3 optional | 22 |
| `food_ideas_v1` (S8) | literal food & cooking | ideas / cognition | categorical, 12↔12 | 24 |
| `frames_v1` (F1) | 5 professional frames × 6 roles | (each other) | categorical, multi-frame | 30 |

The measurement is the paper's: pairwise-cosine RDM per cloud, RSA between the two clouds
through the correspondence $T$, permutation null reshuffling $T$; positions **and** moves
reported in parallel.

## Locked design decisions (do not drift without a note)

1. **One prompt pool per experiment, shared by BOTH clouds.** Precedent: `temp_f1` reusing
   `temp_c1`'s pool. The carrier is then common-mode across the pair being compared. The pool
   is written neutral w.r.t. both domains **and** w.r.t. the relational structure under test —
   a war/argument pool that asked for "two opposing sides" would hand us the correspondence
   for free and the RSA would be an artefact of the carrier.
2. **One negative per prompt per cloud, shared by every concept in that cloud**, positionally
   indexed (`neg[i]` pairs with `train_p[i]`), scrubbing that cloud's own domain only — the
   `h2o_v1` / `bodypart_v1` construction. The two clouds get *separate* negatives; `temp_c1`
   and `h2o_v1` did too and still reached ρ = 0.63–0.72.
   So $v = \text{mean}(\text{scene featuring the concept}) - \text{mean}(\text{same scenes, domain absent})$.
3. **The generator never sees the correspondence.** Each cloud's continuations are independent
   API calls; nothing in a war prompt mentions *criticise*. If it did, the alignment could be
   manufactured at generation time.
4. **Items and $T$ are fixed by the plan**, before any model contact.

## What the corpus audit says to design against

Four defects were found in the shipped `h2o_v1` / `bodypart_v1` data, and all four are
measured in the diagnostics cell below rather than assumed away:

- **Concept-correlated length skew.** `h2o` positives ran **+8.5 words** longer than the
  shared negative, and the skew varied by concept (+2.5 … +14.4) — i.e. it *aliases the very
  axis being measured*. `bodypart_v1` was balanced (66.9 vs 67.1) because both sides carried
  the **same** length instruction. Ours carries `Three or four sentences, no more than 70 words.` on both sides.
- **Cross-concept co-mention.** `boiling_water` positives said "steam" 70/72 times; `sleet`
  said "slush" 60/72. For us this is worse than in `h2o`: a war scene naturally mentions
  defending *and* retreating, an argument passage naturally mentions conceding *and*
  criticising — and **if cloud $A$'s co-mention matrix mirrors cloud $B$'s under $T$, the RSA
  result is a generation artefact, not model geometry.** Handled the `bodypart_v1` way (a
  *soft* "avoid mentioning other X" line, never a hard ban, which would teach an unnatural
  "attack but never defend"), then **measured** — §8e.
- **Formatting asymmetry.** In `bodypart_v1` every defect landed on the positive side: 268
  rows with `**bold**`/newlines, 53 preambles, 9 empty, 2 truncated. All of it folds straight
  into the difference-of-means vector. §8d counts it on both sides.
- **Pool leakage.** `h2o`'s "neutral" pool still contained coffee/kitchen/houseplant hooks;
  `bodypart`'s cycled four openers and let near-duplicates through at Jaccard 0.8.

## What four rounds of real Gemini output changed

Every deviation from the `bodypart_v1` house template below was forced by output I read, not
chosen up front. Each one is marked in the templates cell as well.

| # | Symptom in the samples | Change |
|---|---|---|
| 1 | **Simile, everywhere.** "the kitchen is a hive of quiet efficiency, *much like* a commander executing a Fabian strategy"; "think of it *like* preparing a client for a deposition"; "*just like* the incisions the doctor made". The concept arrived as a *comparison to* the carrier scene — which is precisely the metaphorical blend this design excludes and reserves for the held-out probes. It hit war/`strategy`, war/`retreat`, food/`cook`, food/`stale`, and most of the frame cloud. | New line in `WAVE1_CONT`: *"It must really be there in what is described … never a comparison or a simile for something else (no 'just like', 'much like', 'as if', 'think of it as')…"* |
| 2 | **Assistant-persona weld.** On step-by-step prompts the concept had nowhere to go but the assistant itself: "let me finish *swallowing this bite of my sandwich* so I can speak clearly"; "I need to step away, *my patient* has just arrived". | The same line ("…never something you the assistant are doing or pausing to do"), plus the pools now forbid step-by-step how-to requests. |
| 3 | **How-to pools.** The `bodypart_v1` carrier ("explaining how a common task is done") returned "explain how to thread a sewing machine", which is what created #2. | All four `W.POOLS` moved to a narrative-leaning carrier (narrate / recount / describe / invent), settings widened to include a historical period, an invented world, an organisation, a small town. |
| 4 | **Intimate scenes had no room for a third person.** frames_v1's "a grandfather teaches his granddaughter to graft an apple tree" left nowhere to put a lawyer, so the generator made grandpa the lawyer or reached for a simile. | The frames pool now asks that most requests "involve more than one person, in a public or shared place". |
| 5 | **A war word arrived from the pool.** "…the bottom of the Mariana **Trench**" put a war-ban word into *both* clouds. Common-mode, so harmless to the contrast, but it is the `h2o` "coffee hooks in the neutral pool" defect and it made §8b unreadable. | New `pool_screen(ds, prompt)`: any pool prompt carrying either cloud's own-domain vocabulary is rejected before it is used. |
| 6 | **Markdown and preambles, on the positive side only** (`**bacterium**`, "I can certainly help you with that!"). | The house tail is extended, symmetrically: `Response only - plain prose, no preamble, no markdown.` |
| 7 | **Positives ran long** — mean \|pos − neg\| was 9.9 words at round 1, worse than `h2o`'s 8.5, and a *sentence* cap (round 2) and a soft word budget (round 3) both failed: the model just wrote longer sentences. | `A few sentences.` → `Three or four sentences, no more than 70 words.` on **both** templates, as the last line before `Request:` on both. A **hard cap binds where a soft one did not**: mean \|pos − neg\| fell 9.9 → 12.5 → **3.9** words. |
| 8 | **A refusal.** frames_v1/`contract`: *"I am unable to describe a 1920s lakeside pier while restricting the scene entirely to legal practice."* The frame clause was being read as a constraint on the whole setting. | `W.AVOID_CLAUSE["f1_*"]` now binds the *professional detail*, not the scene: "…The setting itself stays whatever the request asks for." |
| 9 | The importance cloud reached for size words (`immense`, `tiny`, `small`) — the one leak that could hand S3 its result for free. | `W.AVOID_CLAUSE["s3_size"]` widened; the leak went to zero. |

> **Nothing spends API credit until the smoke cell (§6).** `MAX_CALLS = 120` in cell 2 is a
> hard guard; one full pass of §6 costs **118 calls** (4 pools + 24 negatives + 90 positives).
> Re-running §6 after an edit needs `reset_calls()` first.

In [ ]:
import sys, os, asyncio, re, csv, json
from pathlib import Path
import pandas as pd
from IPython.display import Markdown, display
REPO = Path(".")
sys.path.insert(0, str(REPO / "new_pipeline"))
import pipeline as P     # parse_numbered / _classify_prompt / _trigrams / _jaccard

EXT = REPO / "new_pipeline" / "fractal_analysis" / "extension"
sys.path.insert(0, str(EXT))
import wave1_families as W   # items, glosses, locks, pools, scrubs, ban lists

def load_api_key():
    k = os.environ.get("GEMINI_API_KEY")
    if k: return k.strip()
    p = Path.home() / ".api" / "gemini_api_key"
    if p.exists(): return p.read_text().strip()
    raise SystemExit("No Gemini key")

GEN_MODEL = "gemini-3.5-flash"
print("ready | model:", GEN_MODEL, "| key:", "ok" if load_api_key() else "MISSING")
print()
print(W.summary())

In [ ]:
from google import genai
from google.genai import types
MAX_CALLS = 120
_call_count = 0
async def gemini(prompts, *, temperature=1.0, max_output_tokens=1400, concurrency=8, thinking_budget=0):
    global _call_count
    if _call_count + len(prompts) > MAX_CALLS:
        raise RuntimeError(f"would exceed MAX_CALLS={MAX_CALLS}: {_call_count}+{len(prompts)}. reset_calls() or raise it.")
    client = genai.Client(api_key=load_api_key())
    cfg = types.GenerateContentConfig(temperature=temperature, max_output_tokens=max_output_tokens,
        thinking_config=types.ThinkingConfig(thinking_budget=thinking_budget))
    sem = asyncio.Semaphore(concurrency); out = [""] * len(prompts)
    async def one(i, pr):
        async with sem:
            for a in range(4):
                try:
                    r = await client.aio.models.generate_content(model=GEN_MODEL, contents=pr, config=cfg)
                    cand = (r.candidates or [None])[0]
                    parts = (cand.content.parts if (cand and cand.content and cand.content.parts) else []) or []
                    out[i] = "".join(pt.text for pt in parts if getattr(pt, "text", None)); return
                except Exception as e:
                    if a == 3: out[i] = f"[ERROR: {e}]"
                    else: await asyncio.sleep(1.5 * (a + 1))
    await asyncio.gather(*(one(i, p) for i, p in enumerate(prompts)))
    _call_count += len(prompts); print(f"[gemini] +{len(prompts)} calls ({_call_count}/{MAX_CALLS})")
    return out
def reset_calls():
    global _call_count; _call_count = 0; print("reset")

---
# §3 The items and the correspondence $T$

$T$ is fixed **before any model contact** and is what the permutation null reshuffles; the
generator never sees it (§0.3). Read each table across: row $i$ of $A$ corresponds to row $i$
of $B$. The next cell renders them from `wave1_families.py` itself, so what you see is what
will be generated.

## ⚑ Open calls for your inspection (plan v1 §8)

**S3 — size ↔ importance**
- ⚑ rank 1–2 `microscopic/trivial` vs `tiny/negligible`: the trivial↔negligible order is
  defensible either way. The method already handles ties (hail/snow did); say the word and
  they get tie-ranked.
- ⚑ rank 4 uses **`middling`**, swapped in for the plan's earlier "moderate", which reads as
  degree-of-anything rather than degree-of-mattering. `medium-sized ↔ middling` is therefore
  the middle anchor of both ladders.
- Live nuisance is **frequency** (`colossal`, `momentous` rare; `small`, `important` very
  common) — the $|\log f_i - \log f_j|$ row matters most here.

**S7 — war ↔ argument**
- ⚑ **Optional rows 9–11 are currently IN** (`weapon↔evidence`, `stronghold↔core conviction`,
  `ceasefire↔agreeing to disagree`), taking the family from 8 to 11 pairs. Rows 1–8 alone
  clear the resolution floor ($8! = 40{,}320$). Striking any of the three is a one-line edit
  to `_S7_WAR` / `_S7_ARG`; row 11 is the phrasal one whose steerability is most doubtful.
- ⚑ `rebut` has a distant combat etymology (fully debate-domain in modern use); `demolish` is
  destruction rather than strictly military. L&J file both under this metaphor — noted, not
  disqualifying.
- ⚑ This is the pipeline's first **verb-heavy** family. The plan recommends piloting three
  concepts (attack, criticise, strategy) all the way through steering + judge before
  committing. **This notebook is the generation half of that pilot** — it does not steer.

**S8 — food ↔ ideas**
- ⚑ `taste ↔ consider` is the weakest row (row 5); the plan flags it explicitly.
- ⚑ `fresh` is literal-locked to **food** freshness, `novel` to newness-of-idea; they are the
  pair most exposed to the amphibious sense.
- $B$-side etymology audit, stated once: `comprehend` (grasping), `ponder` (weighing) and
  `consider` (star-gazing) are fossil metaphors from *other* source domains. The exclusion
  criterion is **food**-derivation specifically — the confound is source-domain overlap, not
  figurativity in general.

**F1 — professional frames**
- ⚑ The **fifth frame is `teacher`** (proposed, not yet ruled on). It keeps the recipient a
  person who is themselves changed by the intervention, and adds a developmental rather than
  remedial frame. Alternates: accountant, mechanic — both fine, with the entity-split caveat.
- ⚑ **Six roles, not seven.** `OUTCOME` is left out because the dentist cell has no clean
  filler. It is coded in `W.F1_TABLE["OUTCOME"]` and one line switches it on (with dentist
  dropping to 6).
- ⚑ **`patient` duplication.** RECIPIENT repeats across doctor and dentist. Here it is split
  into `patient` / `dental patient` with a teeth-locked gloss, so doctor↔dentist is *not* an
  identity pair; if you would rather keep the honest identity (and note the inflation), make
  the dentist filler `patient` again.
- ⚑ Australian-flavoured cells: `vendor` (the agent's client is the seller; `homebuyer` if you
  want the buyer's side), `dental surgery` (`practice` and `chambers` were rejected as
  amphibious/barrister-specific), `auction` (`open house` / `sale campaign` alternates).
- ⚑ ARTIFACT for dentist is **`dental chart`**, not the plan's `x-ray` — doctors x-ray too.
- ⚑ The second-level F-a question (does frame-pair alignment track intuitive frame
  similarity?) needs a **pre-stated similarity ordering** over the 10 frame pairs. Not written
  yet; it is itself a $T$ and must be fixed before the numbers are looked at.

**Global**
- ⚑ **Move orientation.** S7/S8/F1 have no natural order, and $1-\cos(v_y-v_x,\,v_{y'}-v_{x'})$
  is orientation-sensitive. Convention: fix the canonical order to the row order of these
  tables and orient every move low-index→high-index on *both* sides of $T$; the null
  re-orients under the same rule after each reshuffle. `W.MOVE_ORDER` records it.
- ⚑ **One shared pool per experiment** (§0.1) — a deliberate tightening over `h2o_v1`,
  available because every wave-1 concept is prose-compatible.

In [ ]:
# Render T straight out of wave1_families, so the tables cannot drift from what is generated.
def show(title, df):
    display(Markdown(f"**{title}**"))
    display(Markdown(df.to_markdown(index=False)))

REC = {cl: {r["id"]: r for r in recs} for cl, recs in W.CLOUDS.items()}

for ds in ["size_imp_v1", "war_arg_v1", "food_ideas_v1"]:
    spec = W.EXPERIMENTS[ds]
    A, B = spec["A"], spec["B"]
    rows = []
    for k, (a, b) in enumerate(spec["T"], 1):
        ra, rb = REC[A][a], REC[B][b]
        rows.append({"#": k, f"A · {A}": ra["desc"], "gloss (A)": ra["gloss"],
                     f"B · {B}": rb["desc"], "gloss (B)": rb["gloss"],
                     "⚑": "opt" if k > spec["n_core"] else ""})
    show(f"{ds} — {len(rows)} pairs (core {spec['n_core']})", pd.DataFrame(rows))

# F1 is a role x frame table; every column is a cloud, every row is the index that lines
# the clouds up. Read ACROSS for F-a (frame vs frame), DOWN for F-b (role vs role).
f1 = []
for r in W.F1_ROLES:
    row = {"role": r}
    for f in W.F1_FRAMES:
        cid, desc, _ = W.F1_TABLE[r][f]
        row[f] = desc if cid else "—"
    f1.append(row)
for r in W.F1_ROLES_OPTIONAL:
    row = {"role": r + " ⚑opt"}
    for f in W.F1_FRAMES:
        cid, desc, _ = W.F1_TABLE[r][f]
        row[f] = (desc if cid else "— ⚑") + "  *(off)*"
    f1.append(row)
show("frames_v1 — 5 frames x 6 roles (+1 optional role, currently OFF)", pd.DataFrame(f1))

print(f"{sum(len(v) for v in W.CLOUDS.values())} concepts across "
      f"{len(W.CLOUDS)} clouds / {len(W.EXPERIMENTS)} experiments")

In [ ]:
# ================================================================== CARRIER (prompt pool)
# ONE pool per experiment, shared by BOTH clouds, so the carrier is common-mode across the
# very pair being compared (paper Appendix C; temp_f1 reused temp_c1's pool).
#
# The pool text itself lives in wave1_families.POOLS -- it is DATA, edited freely -- and is
# built on bodypart_v1's ORDINARY-WRITING carrier rather than h2o_v1's "scene description with
# sensory detail". Sensory-scene framing forces flowery prose and, for a domain like war or
# cooking, would drag in the whole domain vocabulary the way it dragged in meronyms.
#
# Two constraints govern each pool's negative clause, and the second one is new here:
#   (a) DOMAIN neutrality -- the pool must lean toward neither cloud; any lean is common-mode
#       across every vector, which is survivable, but a lean toward ONE cloud is not.
#   (b) STRUCTURE neutrality -- the pool must not carry the relational structure under test.
#       A war/argument pool asking for "a scene with two opposing sides" would hand both
#       clouds the correspondence for free and the RSA would be an artefact of the carrier.
#       This is the analogue, one level up, of the paper's "no phase or water word appears in
#       the numeric training text" check.
WAVE1_PROMPTGEN = W.POOLS

# ================================================================== POSITIVE
# bodypart_v1's BP_CONT skeleton with exactly two additions, {lock} and {avoid}:
#
#   line 1  unchanged house wording ("clearly weaving ... so that it features naturally").
#   line 2  {gloss} pins which thing is meant. {lock} is ADDITION 1: it occupies the slot
#           bodypart_v1 used for {syn}, and does the job the synonym line did there -- fix the
#           reading -- but for a SENSE rather than an extension. It is the whole experiment:
#           the war cloud must be literally military and the argument cloud literally verbal,
#           because the famous blended usages ("attack her argument") are held-out probes,
#           not training data.
#           {scale} is ADDITION 3, from the authors' review of the size ladder: for an ORDERED cloud it
#           names the rung's immediate neighbours, and it vanishes for every other cloud, the
#           way bodypart_v1's {syn} line vanished for a concept with no synonym. {syn}
#           delimited a concept from the inside (other words for the same thing); this
#           delimits it from the outside (the nearest things that are NOT it). The defect it
#           answers: Gemini collapsed "sizable" and "colossal" onto one shared intensifier,
#           "massive". See the long note in wave1_families for why naming the neighbours
#           removes a confound rather than adding one, and what it costs.
#   line 3  DEVIATION FROM THE HOUSE TEMPLATE, forced by iteration 1. Three failure modes
#           showed up there and this one line kills all three:
#             SIMILE   -- "the kitchen is a hive of quiet efficiency, much like a commander
#                         executing a Fabian strategy", "think of it like preparing a client
#                         for a deposition". That is the *metaphorical blend* the whole design
#                         excludes: the war cloud must be literal war, not war-as-comparison.
#                         It hit war/strategy, war/retreat, food/cook, food/stale, and most of
#                         the frames cloud.
#             PERSONA  -- "let me finish swallowing this bite of my sandwich so I can speak
#                         clearly", "I need to step away, my patient has just arrived". The
#                         concept welded onto the assistant instead of onto the scene, which
#                         also flips the register from third-person prose to chat.
#             ABSENCE  -- ideas/unoriginal simply did not appear in 2 of 3 completions.
#           {term} is ADDITION 4, also from the authors' review: ask for the concept's own word, not
#           only its sense. Gemini was conveying a rung IMPLICITLY -- a thing read as trivial
#           because an informed reader would see it carried no weight, without "trivial" ever
#           appearing. That is not a drift but a RESTORATION: every existing house dataset has
#           the surface form present in the positive nearly always (bodypart's slug, h2o's
#           desc, temp_c1's numeral). It only fails to happen by itself for gradable
#           adjectives, English having a large near-synonym pool for them -- and h2o shows the
#           same thing, its 4.5% of positives missing the literal token being concentrated in
#           the adjectival water ladder. Deliberately encouraging ("somewhere in the response",
#           any inflection) rather than mandating a form.
#   last    "Three or four sentences, no more than 70 words." -- IDENTICAL on the negative side, and
#           a tightening of the house "A few sentences." forced by the numbers: iteration 1
#           had mean |pos - neg| = 9.9 words, WORSE than h2o's 8.5, with a 20-word within-cloud
#           spread (war/attack +19.0 vs war/strategy -1.7), which is exactly the
#           concept-correlated skew that aliases the measured axis. Iteration 2 capped the
#           SENTENCE count and the model simply wrote longer sentences (9.6), so iteration 3
#           states the budget in the unit the defect is measured in, iteration 4 moves it to
#           the LAST line so both templates carry it immediately before "Request:", and makes
#           the budget a CAP ("no more than"), since a soft "about" did not bind.
#           Section 8a. It does not close the gap: the positive has a whole extra event to
#           deliver, so a residual cloud-level offset is intrinsic to this construction -- see
#           the noise floor in 8a, and the note in section 9.
#   line 4  the house soft co-mention line, with "body parts" parameterised as {kind} (we have
#           twelve clouds where bodypart_v1 had one). SOFT on purpose: hard-banning the other
#           concepts would teach "attack but never defend", an unnatural contrast that could
#           hide the real effect. We do not suppress co-mention, we MEASURE it (section 8e).
#   line 5  {avoid} is ADDITION 2, and it is the load-bearing control for the whole cross-family
#           claim, which is why its effect is invisible when it works. Each cloud is given the
#           OTHER cloud's vocabulary to stay off: the importance cloud may not say "a massive
#           decision", the argument cloud may not say "attack her claim". Without it the two
#           clouds would share surface strings, and an RSA agreement between them could be read
#           straight off that shared vocabulary rather than off the model's geometry -- the
#           metaphor would be one we handed the generator. The words it withholds are precisely
#           the amphibious ones English already lexicalised ("a weighty question", "a big
#           decision"), and they are not discarded: they are the held-out figurative probes
#           (W.PROBES), projected after the alignment is established on disjoint vocabulary.
#           Section 8b counts what got through; the smoke run leaked 4 words in 30 concepts.
#   tail    "plain prose, no preamble, no markdown" extends the house "Response only - no
#           preamble." -- iteration 1 put **bold** on the concept word in the positives only
#           (size/microscopic, doctor/illness) and preambles on 4 of 26, and every such defect
#           lands on ONE side of pos - neg. Extended on BOTH templates, so it stays symmetric.
WAVE1_CONT = """Write one short, natural response an AI assistant would give to the request below, clearly weaving {desc} into it so that it features naturally.
Here {desc} means: {gloss}. {lock}{scale}
Use the word "{term}" itself somewhere in the response rather than only implying it, and let it really be there in what is described - part of the scene or of the answer itself, never a comparison or a simile for something else (no "just like", "much like", "as if", "think of it as"), and never something you the assistant are doing or pausing to do.
Avoid mentioning other {kind} that are not {desc}, that includes both {kind} that are closely related to {desc}, and those that are very different.
{avoid}
Three or four sentences, no more than 70 words.

Request: {prompt}

Response only - plain prose, no preamble, no markdown."""

# ================================================================== NEGATIVE (shared per cloud)
# Generated ONCE PER PROMPT and shared by every concept in the cloud, so every vector in the
# cloud subtracts the SAME baseline over the SAME prompts and the only systematic difference
# is the concept. {scrub} removes that cloud's own domain and nothing else: what is left is
# the cloud's carrier (war-ness, food-ness), which the move construction then differences away.
# Note the length instruction rides at the end of the scrub line -- the same words, in the
# same last position, as on the positive side, which is what keeps the lengths comparable.
WAVE1_CONT_ABSENT = """Write one short, natural response an AI assistant would give to the request below.
{scrub} Three or four sentences, no more than 70 words.

Request: {prompt}

Response only - plain prose, no preamble, no markdown."""


def render_wave1_promptgen(ds, n):
    return WAVE1_PROMPTGEN[ds].format(n=n)

def render_wave1_absent(cloud, prompt):
    return WAVE1_CONT_ABSENT.format(scrub=W.SCRUB[cloud], prompt=prompt)

def render_wave1_cont(rec, prompt):
    return WAVE1_CONT.format(desc=rec["desc"], gloss=rec["gloss"], lock=rec["lock"],
                             scale=W.scale_note(rec), term=W.head_term(rec),
                             kind=W.CLOUD_KIND[rec["cloud"]],
                             avoid=W.AVOID_CLAUSE[rec["avoid_key"]], prompt=prompt)

# The pool screen is an ADDITION to the house filter, and iteration 1 is why. The war/argument
# pool returned "...at the bottom of the Mariana TRENCH", and "trench" is on the war ban list,
# so every argument-cloud completion on that prompt inherited a war word from the CARRIER. It
# is common-mode (both clouds got it), so it does not fake the contrast -- but it is exactly
# the h2o "coffee/kitchen hooks in the neutral pool" defect, and it makes the leak check in
# section 8b unreadable. Cheapest fix: reject any pool prompt carrying either cloud's own
# domain vocabulary, before it is ever used.
_POOL_BAN = {ds: sorted({w for cl in cls for w in W.OWN_BAN[cl]})
             for ds, cls in W.EXPERIMENT_CLOUDS.items()}

def pool_screen(ds, prompt):
    """[] if the prompt is clean for this experiment, else the domain words it carries."""
    pat = re.compile(r"\b(" + "|".join(re.escape(w) for w in
                     sorted(_POOL_BAN[ds], key=len, reverse=True)) +
                     r")(s|es|ed|d|ing|ion|ly)?\b", re.I)
    return sorted({m.group(1).lower() for m in pat.finditer(prompt)})

def filter_prompts(cands_, n, th=0.8, ds=None):
    kept, grams = [], []
    for p in cands_:
        if len(kept) >= n: break
        if P._classify_prompt(p)[0] == "hard": continue
        if ds is not None and pool_screen(ds, p): continue
        g = P._trigrams(p)
        if max((P._jaccard(g, kg) for kg in grams), default=0) >= th: continue
        kept.append(p); grams.append(g)
    return kept

print("templates loaded | carrier=ordinary-writing | pools:", list(WAVE1_PROMPTGEN))
_p = "Explain how someone goes about repotting a houseplant that has outgrown its pot."
print("\n--- exactly what gets sent, POSITIVE, war__attack ---\n")
print(render_wave1_cont(W.CLOUDS["war"][0], _p))
print("\n--- exactly what gets sent, NEGATIVE (shared by the whole war cloud) ---\n")
print(render_wave1_absent("war", _p))

In [ ]:
# ============================== SMOKE RUN ==============================
# One pool call per experiment -> filter to 3 prompts -> one shared negative batch per cloud
# -> positives for a deliberately SPANNING subset. For the two-cloud families the picks are
# CORRESPONDENCE-PAIRED, so section 7 puts a↔T(a) side by side and you can read the alignment
# question directly off the prose. For frames the pick is one WHOLE frame plus the two
# matching roles from a second frame, which is the F-a comparison in miniature.
#
# Cost: 4 pools + 24 negatives + 90 positives = 118 calls, against MAX_CALLS=120.
N_SMOKE   = 3
POOL_ASK  = 24

SMOKE_PICKS = {
    # rank 1 / 5 / 9 -- both ends of the ladder and the middle anchor
    "size_imp_v1":   [("size", "microscopic"), ("importance", "trivial"),
                      ("size", "sizable"),     ("importance", "noteworthy"),
                      ("size", "colossal"),    ("importance", "momentous")],
    # the three the plan names for the verb pilot, plus the retreat/back-down pair, which is
    # where the "argument scene borrows war vocabulary" risk is highest
    "war_arg_v1":    [("war", "attack"),   ("argument", "criticise"),
                      ("war", "retreat"),  ("argument", "back_down"),
                      ("war", "strategy"), ("argument", "reasoning")],
    # a verb pair, a bodily pair and an adjective pair -- the three grammatical shapes present
    "food_ideas_v1": [("food", "cook"),    ("ideas", "devise"),
                      ("food", "swallow"), ("ideas", "accept"),
                      ("food", "stale"),   ("ideas", "unoriginal")],
    # two WHOLE frames, role for role -- this is the F-a comparison in miniature, and the
    # only pick that makes the cross-cloud co-mention check in section 8e computable for F1
    "frames_v1":     [("doctor", r["id"]) for r in W.CLOUDS["doctor"]] +
                     [("lawyer", r["id"]) for r in W.CLOUDS["lawyer"]],
}

def _clouds_of(ds):
    seen = []
    for cl, _ in SMOKE_PICKS[ds]:
        if cl not in seen: seen.append(cl)
    return seen

async def smoke(ds):
    raw = (await gemini([render_wave1_promptgen(ds, POOL_ASK)],
                        temperature=1.0, max_output_tokens=1600, concurrency=4))[0]
    cands  = P.parse_numbered(raw, POOL_ASK)
    dirty  = [(c, pool_screen(ds, c)) for c in cands]
    prompts = filter_prompts(cands, N_SMOKE, ds=ds)
    n_dirty = sum(1 for _, h in dirty if h)
    print(f"[{ds}] pool: {len(cands)} parsed | {n_dirty} rejected by the domain screen "
          f"-> {len(prompts)} kept")
    for c, h in dirty:
        if h: print(f"    screened out {h}: {c[:88]}")
    neg = {}
    for cl in _clouds_of(ds):
        neg[cl] = await gemini([render_wave1_absent(cl, p) for p in prompts],
                               temperature=0.7, max_output_tokens=400, concurrency=12)
    pos = {}
    for cl, cid in SMOKE_PICKS[ds]:
        pos[(cl, cid)] = await gemini([render_wave1_cont(REC[cl][cid], p) for p in prompts],
                                      temperature=0.7, max_output_tokens=400, concurrency=12)
    return dict(prompts=prompts, pool_raw=raw, pool_cands=cands, neg=neg, pos=pos)

RES = {}
for _ds in ["size_imp_v1", "war_arg_v1", "food_ideas_v1", "frames_v1"]:
    RES[_ds] = await smoke(_ds)

# tidy frame: one row per generated completion
_rows = []
for ds, R in RES.items():
    for cl, texts in R["neg"].items():
        for i, t in enumerate(texts):
            _rows.append(dict(ds=ds, cloud=cl, concept="(absent)", polarity="negative",
                              prompt_idx=i, prompt=R["prompts"][i], text=t,
                              n_words=len(t.split())))
    for (cl, cid), texts in R["pos"].items():
        for i, t in enumerate(texts):
            _rows.append(dict(ds=ds, cloud=cl, concept=cid, polarity="positive",
                              prompt_idx=i, prompt=R["prompts"][i], text=t,
                              n_words=len(t.split())))
DF = pd.DataFrame(_rows)
print(f"\n{len(DF)} completions | {DF.n_words.mean():.1f} words mean")
display(DF.groupby(["ds", "polarity"]).n_words.agg(["count", "mean"]).round(1))

In [ ]:
# ============================== READ THE PROSE ==============================
# Grouped BY SHARED PROMPT (bodypart_v1 cell-16 format): the same request, then that cloud's
# ONE shared negative, then each concept's positive. A-cloud and B-cloud are shown as two
# blocks under the same prompt with the correspondence arrow "<- T <-" on the B side, so the
# question the RSA will ask ("is B's internal arrangement A's, relabelled?") is legible here.
def review(ds, max_prompts=None):
    R = RES[ds]; spec = W.EXPERIMENTS[ds]
    Tmap = {b: a for a, b in spec["T"]} if spec["A"] else {}
    picks = SMOKE_PICKS[ds]
    display(Markdown(f"# {ds}"))
    for j, p in enumerate(R["prompts"][:max_prompts or len(R["prompts"])]):
        out = [f"### shared prompt {j}: _{p}_", ""]
        for k, cl in enumerate(_clouds_of(ds)):
            side = "A" if k == 0 else "B"
            out += ["", f"**{side} · {cl} cloud**", "",
                    f"- **neg (shared, {cl}-absent):** {R['neg'][cl][j]}"]
            for c2, cid in picks:
                if c2 != cl: continue
                rec = REC[cl][cid]
                tag = rec["desc"]
                if cid in Tmap:
                    tag += f"  ⟵T⟵ *{REC[spec['A']][Tmap[cid]]['desc']}*"
                elif "role" in rec:
                    tag += f"  *[{rec['role']}]*"
                out.append(f"- **pos · {cid}** ({tag}): {R['pos'][(cl, cid)][j]}")
        display(Markdown("\n".join(out))); display(Markdown("---"))

review("size_imp_v1")

In [ ]:
review("war_arg_v1")

In [ ]:
review("food_ideas_v1")

In [ ]:
review("frames_v1")

In [ ]:
# ============================== DIAGNOSTICS ==============================
# Every number here targets one of the four defects found in the shipped corpus (section 0).
# At N_SMOKE=3 prompts these are INDICATIVE, not decisive -- they are here to catch a template
# that is obviously wrong before we pay for 72 prompts x 94 concepts.
import numpy as np

SUF = r"(s|es|ed|d|ing|ion|ly)?"
def _rx(words):
    """Word-boundary on BOTH ends, plus a bounded inflectional tail. Both ends matter: a
    naive prefix match flags 'warranted' for 'war' and 'litt-' for 'little'."""
    alts = "|".join(re.escape(w) for w in sorted(set(words), key=len, reverse=True))
    return re.compile(rf"\b({alts}){SUF}\b", re.I)

def _hits(text, words):
    return sorted({m.group(1).lower() for m in _rx(words).finditer(text)})

def cap(s): display(Markdown(s))

In [ ]:
# ---- 8a. LENGTH BALANCE ------------------------------------------------------------
# The h2o defect: positives +8.5 words over the shared negative, and the skew varying by
# concept (+2.5 ... +14.4), so length ALIASED the axis being measured. Both sides here carry
# the identical length instruction; this checks whether that worked, and how much of what
# is left is sampling noise at three prompts.
rows = []
for ds, R in RES.items():
    for (cl, cid), texts in R["pos"].items():
        neg = R["neg"][cl]
        wp = np.array([len(t.split()) for t in texts], float)
        wn = np.array([len(t.split()) for t in neg], float)
        rows.append(dict(ds=ds, cloud=cl, concept=cid, pos=wp.mean(), neg=wn.mean(),
                         # SE of the paired per-prompt difference: the noise floor on `delta`
                         se=(wp - wn).std(ddof=1) / np.sqrt(len(wp))))
LEN = pd.DataFrame(rows)
LEN["delta"] = LEN["pos"] - LEN["neg"]
LEN["dev"] = LEN["delta"] - LEN.groupby("cloud")["delta"].transform("mean")

cap("**8a · length balance — positive vs its cloud's shared negative (words)**")
display(LEN.groupby(["ds", "cloud"])[["pos", "neg", "delta"]].mean().round(1))
cap("A cloud-level offset is survivable: it is common-mode across every vector in the cloud "
    "and it differences away in the move construction. What is NOT survivable is a "
    "**concept-correlated** offset, because that aliases the axis being measured. So each "
    "concept's deviation from its own cloud's mean is put against its own standard error "
    "(the SE of the paired per-prompt difference). At 3 prompts that noise floor is ~10 "
    "words, so almost nothing here can be significant — the column is a template for the "
    "72-prompt run, where the SE falls by a factor of ~5.")
LEN["flag"] = np.where(LEN.dev.abs() > 2 * LEN.se.replace(0, np.nan), "⚑", "")
display(LEN.sort_values("dev", key=abs, ascending=False)
           [["ds", "cloud", "concept", "pos", "neg", "delta", "dev", "se", "flag"]]
           .round(1).head(20))
print(f"\ncorpus-wide mean |delta| = {LEN.delta.abs().mean():.1f} words "
      f"(h2o_v1 was +8.5; bodypart_v1 was -0.2)")
print(f"median per-concept SE at this sample size = {LEN.se.median():.1f} words; "
      f"{(LEN.flag == '⚑').sum()}/{len(LEN)} concepts exceed 2 SE")

In [ ]:
# ---- 8b. CROSS-DOMAIN VOCABULARY LEAK ----------------------------------------------
# The {avoid} clause forbids the OTHER family's vocabulary. If it leaks, cross-family
# agreement could be handed to us by shared surface strings rather than by geometry.
# HARD  = W.BAN[avoid_key]      -- instructed, so a hit is a template failure.
# SOFT  = W.BAN_SOFT[avoid_key] -- amphibious probe words, NEVER instructed; reported only,
#                                  because these are the words the metaphor itself lexicalised.
rows = []
for ds, R in RES.items():
    for (cl, cid), texts in R["pos"].items():
        key = REC[cl][cid]["avoid_key"]
        hard = [h for t in texts for h in _hits(t, W.BAN[key])]
        soft = [h for t in texts for h in _hits(t, W.BAN_SOFT[key])]
        rows.append(dict(ds=ds, cloud=cl, concept=cid, n=len(texts),
                         hard=len(hard), hard_words=", ".join(sorted(set(hard))) or "-",
                         soft=len(soft), soft_words=", ".join(sorted(set(soft))) or "-"))
LEAK = pd.DataFrame(rows)
cap("**8b · cross-domain leak in the POSITIVES** (hard = instructed against; soft = amphibious, reported only)")
cap("_Known homonyms that this check over-reports, because the ban lists are word lists and "
    "not sense lists: **trench** (an ocean trench is not a war trench), **filling** (filling out a form is not a dental filling) and **vendor** (a "
    "market stall-holder is not a property vendor). The bias is deliberate -- a QC check "
    "should over-report._")
display(LEAK.sort_values("hard", ascending=False))
print(f"HARD leaks: {int(LEAK.hard.sum())} across {int((LEAK.hard>0).sum())}/{len(LEAK)} concepts")
print(f"SOFT hits : {int(LEAK.soft.sum())} across {int((LEAK.soft>0).sum())}/{len(LEAK)} concepts")

In [ ]:
# ---- 8c. NEGATIVE PURITY -----------------------------------------------------------
# The one failure mode that corrupts every vector in a cloud at once, since all of them
# subtract this same pool (bodypart_v1 cell 17, generalised to per-cloud negatives).
rows = []
for ds, R in RES.items():
    for cl, texts in R["neg"].items():
        for j, t in enumerate(texts):
            h = _hits(t, W.OWN_BAN[cl])
            rows.append(dict(ds=ds, cloud=cl, prompt_idx=j,
                             status="LEAK" if h else "clean", words=", ".join(h) or "-"))
NEG = pd.DataFrame(rows)
cap("**8c · does each cloud's shared negative mention its OWN domain?**")
display(NEG)
print(f"{(NEG.status=='LEAK').sum()}/{len(NEG)} negatives leak their own domain.")
print("A little is survivable (it is common-mode across every concept in the cloud); a lot")
print("means the negative is not a clean 'domain absent' baseline -- tighten W.SCRUB[cloud].")

In [ ]:
# ---- 8d. FORMATTING --------------------------------------------------------------
# In bodypart_v1 EVERY formatting defect landed on the positive side (268 markdown/newline,
# 53 preambles, 9 empty, 2 truncated). Anything one-sided folds straight into pos - neg.
PRE = re.compile(r"^\s*(sure|certainly|of course|absolutely|okay|ok|here'?s|here is|"
                 r"i can|i'?d be happy|happy to)\b", re.I)
REF = re.compile(r"^\s*(i cannot|i can'?t|i'?m unable|i am unable|as an ai|i won'?t)\b", re.I)
def defects(t):
    d = []
    if not t.strip() or t.startswith("[ERROR"): d.append("empty/error")
    if "**" in t or "##" in t: d.append("markdown-bold")
    if re.search(r"(?<!\*)\*[^*\n]+\*(?!\*)|(?<![\w_])_[^_\n]+_(?![\w_])", t): d.append("markdown-italic")
    if "\n" in t.strip(): d.append("newline")
    if PRE.match(t): d.append("preamble")
    if REF.match(t): d.append("refusal")
    if t.strip() and t.strip()[-1] not in '.!?"”’': d.append("truncated?")
    return d
DF["defects"] = DF.text.map(lambda t: ",".join(defects(t)))
cap("**8d · formatting defects by polarity** (the asymmetry is what matters, not the count)")
fmt = (DF.assign(bad=DF.defects != "")
         .groupby(["ds", "polarity"]).bad.agg(["sum", "count"]))
fmt["rate"] = (fmt["sum"] / fmt["count"]).round(3)
display(fmt)
bad = DF[DF.defects != ""]
if len(bad):
    display(bad[["ds", "cloud", "concept", "polarity", "prompt_idx", "defects"]])
    for _, r in bad.head(6).iterrows():
        print(f"\n[{r.ds}/{r.cloud}/{r.concept}/{r.polarity}] {r.defects}\n  {r.text[:220]!r}")
else:
    print("no formatting defects in the smoke sample.")

In [ ]:
# ---- 8e. CO-MENTION, WITHIN AND ACROSS CLOUDS --------------------------------------
# THE ONE THAT MATTERS. C[a,b] = fraction of concept a's positives that mention concept b.
# Within a cloud, co-mention is expected and is NOT suppressed (a war scene that mentions
# retreating is a natural war scene; hard-banning it would teach "attack but never defend").
#
# The danger is the CROSS-cloud version: if cloud A's co-mention matrix rank-correlates with
# cloud B's THROUGH the correspondence T, then the two clouds were written with the same
# internal wiring at generation time, and any RSA agreement we later measure is an artefact of
# how the text was produced, not a fact about the model's geometry. Near zero is what we want.
def comention(R, picks, cloud):
    """Rows = the concepts actually sampled (only those have text). Columns = EVERY
    concept in the cloud, in T order.

    Columns must be the whole cloud, not the smoke subset. With subset columns the
    matrix misses the commonest case outright -- "a sizable cabinet packed with tiny
    vials", where `tiny` is a rung that happens not to be in the subset -- and then
    reads as a reassuring flat zero. Rows stay the subset because only those concepts
    were generated; the row count is what the full run fixes, not the column count."""
    rows = [cid for cl, cid in picks if cl == cloud]
    cols = list(W.MOVE_ORDER[cloud])
    pats = {cid: _rx(W.concept_terms(REC[cloud][cid])) for cid in cols}
    C = np.zeros((len(rows), len(cols)))
    for i, a in enumerate(rows):
        texts = R["pos"][(cloud, a)]
        for j, b in enumerate(cols):
            if a == b: continue
            C[i, j] = np.mean([bool(pats[b].search(t)) for t in texts])
    return C, rows, cols

def _spearman(x, y):
    x, y = pd.Series(x).rank(), pd.Series(y).rank()
    if x.std() == 0 or y.std() == 0: return float("nan")
    return float(np.corrcoef(x, y)[0, 1])

for ds, R in RES.items():
    picks = SMOKE_PICKS[ds]
    cls = _clouds_of(ds)
    cap(f"**8e · {ds}**")
    Cs = {}
    for cl in cls:
        C, rows, cols = comention(R, picks, cl)
        Cs[cl] = (C, rows, cols)
        display(Markdown(f"*within-cloud co-mention, {cl} "
                         f"(row = sampled concept, column = every concept in the cloud)*"))
        display(pd.DataFrame(C, index=rows, columns=cols).round(2))
    if len(cls) == 2:
        (CA, ra, ca), (CB, rb, cb) = Cs[cls[0]], Cs[cls[1]]
        # rows and columns are both in T order, so cell (i, j) of A corresponds to
        # cell (i, j) of B; drop only the self cells, which are structurally zero.
        if CA.shape == CB.shape:
            off = np.array([[ca[j] != ra[i] for j in range(len(ca))] for i in range(len(ra))])
            off &= np.array([[cb[j] != rb[i] for j in range(len(cb))] for i in range(len(rb))])
            r = _spearman(CA[off], CB[off])
            if np.isnan(r):
                sa, sb = CA[off].sum(), CB[off].sum()
                both = CA[off].std() == 0 and CB[off].std() == 0
                print(f"cross-cloud co-mention rank-correlation under T: UNDEFINED "
                      f"(A sum={sa:.2f}, B sum={sb:.2f})")
                if both:
                    print("  Neither cloud has any off-diagonal co-mention, so there is no "
                          "wiring to share.\n  That is the best available outcome for this "
                          "check.")
                else:
                    lo, hi = (cls[0], cls[1]) if CA[off].std() == 0 else (cls[1], cls[0])
                    print(f"  Undefined because '{lo}' is entirely flat -- a constant has no "
                          f"rank order.\n  The two clouds therefore cannot SHARE co-mention "
                          f"wiring, which is what this check is for.\n  But '{hi}' is not "
                          f"flat, and that is its own (smaller) issue: its positives do reach "
                          f"for\n  a contrasting concept from the same cloud, so part of its "
                          f"internal geometry is\n  written in at generation time. Read its "
                          f"matrix above.")
            else:
                print(f"cross-cloud co-mention rank-correlation under T: rho = {r:+.2f} "
                      f"(n = {int(off.sum())} off-diagonal cells)")
            print("  A HIGH value here would mean the two clouds were written with the same")
            print("  internal co-mention wiring, so a later RSA agreement would be a GENERATION")
            print("  ARTEFACT rather than model geometry. Near 0 is what we want.")
    print("Indicative only at 3 prompts per concept -- the full run has 72.")

---
# §9 What to look at, and what to decide

## Where the final round landed

| check | result | reference |
|---|---|---|
| length balance (§8a) | mean \|pos − neg\| = **3.9 words**; per-cloud +0.9 … +7.9; **1/30** concepts beyond 2 SE | `h2o_v1` +8.5, `bodypart_v1` −0.2 |
| cross-domain leak (§8b) | **4 hard hits / 30 concepts**, and 2 of the 4 are the known homonyms | round 1 had leaks in three of four families |
| negative purity (§8c) | **0 / 24** negatives mention their own domain | — |
| formatting (§8d) | **0 defects on either side** — no markdown, no preamble, no refusal, no truncation | `bodypart_v1` had 268 markdown + 53 preambles, all on the positive |
| co-mention, S-families (§8e) | essentially **flat**: one non-zero off-diagonal cell in six matrices | `h2o` had `boiling_water`→steam at 70/72 |
| co-mention, `frames_v1` (§8e) | cross-frame **ρ = +0.45** under $T$ — the one number that is not clean | see (b) below |

## Read the prose in §7 against these, in this order

1. **Is the war positive a genuine literal military scene, or is it forced?** This was the
   single biggest risk in the design, and it has two parts. *Literalness* is now fine — after
   the anti-simile line, every war positive is a real armed action, not a metaphor for the
   carrier scene. *Naturalness* is the residual: because the pool is domain-neutral, almost
   every war positive uses the same device — **the carrier scene continues while the war
   happens outside the window** ("the quiet dawn in the bakery is shattered as a military
   attack commences on the nearby outpost"). It is coherent fiction rather than absurd, but it
   is a stereotyped construction, and it means the war content sits as an additive block on
   top of a shared carrier rather than being integrated into it. For a difference-of-means
   vector that is arguably the *good* case (the carrier is exactly what the negative
   subtracts), but it is a decision, not an accident — see the options below. The same device
   appears in the food cloud ("the ship's cook was frying fish on the deck above, oblivious to
   the standoff below") and in about a third of the frame positives.
2. **Does the argument positive avoid war vocabulary, and the war positive avoid argument
   vocabulary?** §8b is the count and it is now zero for `war_arg_v1`; the prose is the judge
   of whether the avoidance made the text stilted. It did not — the argument cloud reads as
   ordinary shop-floor and workshop disagreement.
3. **Do the food positives stay literal** — an actual kitchen, actual eating — rather than
   sliding into "food for thought"? They do: sausages on a gas grill, a mouthful of rye bread,
   a hardened kaiser roll. `digest`, `chew` and `swallow` are the ones to keep watching in the
   full run.
4. **Are `size` and `importance` clearly distinct**, or does "a colossal statue" blur into "a
   momentous decision"? They are distinct, and cleanly so: `colossal` gets granite obelisks and
   suspension bridges, `momentous` gets a sealed envelope and a midnight meeting. §8b confirms
   no shared vocabulary.
5. **Do the frame positives stay inside one profession?** Mostly — and the one refusal is
   gone. The residue is `property`: a lawyer scene naturally reaches for a *property dispute*,
   which is real-estate-frame vocabulary (2 of 30 concepts in the final run). That is a genuine
   overlap between two of the five frames, not a template bug, and it is worth deciding whether
   to instruct against it, because `lawyer` and `agent` are one of the ten F-a frame pairs.
6. **Are positive and negative the same length and register?** Register, yes. Length, **no** —
   see the caveat below.

## Three things the diagnostics say that you should decide on

**(a) Length balance is fixed, but only by a hard cap, and it needs re-reading at n=72.**
This took four rounds. `A few sentences.` (the house wording) gave mean \|pos − neg\| = 9.9
words — *worse* than `h2o`'s 8.5. Capping **sentences** made it worse still (12.5): the model
simply wrote longer sentences. A soft word budget ("about 70 words") did not bind either. A
**hard** cap ("no more than 70 words"), identical on both templates and in the same last
position on both, brought it to **3.9 words** — per-cloud offsets now run +0.9 to +7.9, with
the frame clouds effectively balanced (+1.0, +0.9). Two things to keep in view. A cloud-level
offset is survivable anyway: it is common-mode across every vector in the cloud and differences
away in the move construction. The **concept-correlated** part is the one that aliases the
measured axis and the one that actually broke `h2o` — here only 1 of 30 concepts exceeds two
standard errors, and the median SE at three prompts is 2.7 words. **Re-read that column at 72
prompts, where the SE falls by about a factor of five and a real per-concept effect would
finally be visible.** If one shows up there, the honest fix is to truncate every completion to
a common word count at conversion time, not to add another instruction.

**(b) `frames_v1` has real co-mention structure, and it is partly shared across frames.**
§8e: the doctor and lawyer clouds both co-mention their PRACTITIONER far more than anything
else (`clinic`→doctor, `prescription`→doctor, `representation`→lawyer, `contract`→lawyer), and
the cross-frame rank correlation of the two co-mention matrices under $T$ came out at
**ρ = +0.34** and then **+0.45** on the last two rounds (30 off-diagonal cells, three prompts —
noisy, but the mechanism is structural and will not wash out). That is not nothing. It means part of any F-a agreement we later measure could
have been written into the text at generation time rather than being a fact about the model's
geometry. Three responses, and one has to be chosen before the numbers are looked at:
*(i)* keep the soft avoid line as it is, measure $C$ on the full 72-prompt run, and report
`RSA(PM_A, PM_B | C)` — the partial — alongside the raw figure, which is the `bodypart_v1`
answer and the one this notebook is set up for; *(ii)* harden the avoid line for the frames
only, at the cost of unnatural text ("a prescription, but never say who wrote it"); *(iii)*
accept it and argue it in the write-up. The three S-families have **no** such problem — their
within-cloud co-mention came out at essentially zero, so there is no structure to correlate.

**(c) The one-shared-pool decision survives, but only because the pool was rewritten.**
On the original `bodypart_v1` carrier it did not: how-to prompts drove the concept into the
assistant's persona. On the narrative carrier the same pool serves both clouds of every
experiment without either cloud being favoured, which keeps the paper's common-mode-carrier
property. If a family later proves it cannot live on a shared pool, the fallback is a
per-cloud pool, and that must be declared, because it removes exactly the property that makes
the two clouds comparable.

## Then decide, before the full run

- ⚑ every open call listed in §3: S7's optional rows 9–11, the fifth frame, the OUTCOME role,
  the `patient` duplication, the Australian cells, the frame-similarity ordering for the
  second-level F-a question, and the move-orientation convention;
- ⚑ (a), (b) and (c) above;
- ⚑ whether S7 gets the three-concept steering pilot (attack, criticise, strategy) before the
  other three families are generated, as `detailed plan v1.md` §3.2 recommends. Generation is
  clearly fine now; steerability of a verb-heavy family is still untested.

## Runbook once approved

The generator does not exist yet. It will follow `generate_bodypart_v1.py` exactly: hold no
prompt text of its own, and `exec()` this notebook's cells by marker
(`import pipeline as P`, `async def gemini`, `WAVE1_PROMPTGEN =`) — which is why the templates
cell must stay a single cell whose only trailing code is a safe demo print. It should also
carry §8's checks over the full output and drop rows flagged by 8d (one preamble and one
refusal appeared in 114 completions here, ~2%).

```bash
PY=python
EXT=fractal_analysis/extension

# 0. re-generate this notebook after editing the builder
$PY $EXT/build_wave1_workshop.py
jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.kernel_name=steering --ExecutePreprocessor.timeout=1800 \
    $EXT/wave1_workshop.ipynb

# 1. generate (one dataset at a time; --dry prints specs and spends nothing)
$PY $EXT/generate_wave1.py --dataset war_arg_v1 --dry
$PY $EXT/generate_wave1.py --dataset war_arg_v1 --n-prompts 4      # smoke
$PY $EXT/generate_wave1.py --dataset war_arg_v1                    # full, n=72

# 2. convert -> axbench -> configs -> SLURM, per dataset
$PY convert_to_axbench.py \
    --vectors inputs/war_arg_v1_vectors.jsonl \
    --raw-dir data/raw/war_arg_v1 \
    --out-dir data/axbench/war_arg_v1
$PY hypothesis_test/make_configs.py war_arg_v1
DATASETS="war_arg_v1" LAYERS="0 5 10 15 20" bash slurm/submit_temperature_v1.sh
```

Layers are the house default 5/10/15/20 **plus layer 0**, so every wave-1 family contributes to
the layer-curve-with-embedding-baseline figure the paper plan already mandates. Held-out probes
(`W.PROBES`) are trained with the same recipe, excluded from every matrix, and projected
afterwards, Figure-6 style. The judge needs no rubric change: each concept's sense lock rides
inside the `concept` string that `gemma4_judge.py` already takes as free text (see
`W.CLOUDS[...][i]["concept"]`).